# Method 3: ESM2 Foundation Model Embeddings for Protein Function Retrieval

**Project:** Benchmarking Protein Function Retrieval — BLAST vs k-mer vs ESM2  
**Course:** COMPSCI 690U — Final Project  
**Team:** Sameeksha Bhatia · Shreya Arathi · Sivaraaman Balakrishnan · Sergey Semibratov

---

## Overview

This notebook implements **Method 3**: dense retrieval using [ESM2](https://huggingface.co/facebook/esm2_t33_650M_UR50D) protein language model embeddings. We evaluate three model scales (35M, 150M, 650M parameters) on two DGEB retrieval tasks:

| Task | Queries | Corpus |
|------|---------|--------|
| **Arch Retrieval** | 2,343 archaeal proteins | 9,229 bacterial proteins |
| **Euk Retrieval** | 311 eukaryotic proteins | 3,202 bacterial proteins |

**Primary metric:** MAP@5  
**Secondary metrics:** nDCG@5, Recall@5  
**Reference upper bound:** ESM2-3B MAP@5 = 0.313 (Arch), 0.357 (Euk) — from DGEB paper

### Notebook Sections
1. Environment setup & GPU verification  
2. Install & import dependencies  
3. Load DGEB datasets (Arch + Euk)  
4. ESM2 embedding extraction (middle + last encoder layers)  
5. Cosine similarity retrieval (top-5)  
6. Evaluation: MAP@5, nDCG@5, Recall@5  
7. Twilight zone analysis (requires BLAST % identity from Method 1)  
8. Visualization & results table  

> **Runtime note:** ESM2-650M on A100 takes ~20 min for Arch corpus. Run Section 4 cells sequentially and use the checkpoint save/load pattern to avoid recomputation on reconnects.

---
## Section 1: Environment Setup & GPU Check

In [1]:
import subprocess, sys, os

# Verify GPU availability and VRAM
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                           '--format=csv,noheader'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print('GPU detected:', gpu_info.stdout.strip())
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU (A100 recommended).')
    print('ESM2-650M requires ~14 GB VRAM; use A100 on Colab Pro for reliable execution.')

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU detected: NVIDIA A100-SXM4-40GB, 40960 MiB, 40442 MiB
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


---
## Section 2: Install & Import Dependencies

In [2]:
# Install all required packages
# dgeb provides the benchmark datasets and evaluation utilities
%pip install -q dgeb transformers accelerate datasets scikit-learn pandas matplotlib seaborn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 8.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 31.1 MB/s eta 0:00:00


In [3]:
import os
import json
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, EsmModel

from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Directory for saving embeddings checkpoints (persists across Colab sessions via Drive)
CHECKPOINT_DIR = '/content/esm2_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint directory: {CHECKPOINT_DIR}')

Using device: cuda
Checkpoint directory: /content/esm2_checkpoints


In [4]:
# Mount Google Drive to persist checkpoints across session reconnects
# Highly recommended for ESM2-650M which takes ~20 min to embed
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/esm2_checkpoints_690U'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f'Drive mounted. Checkpoints will be saved to: {CHECKPOINT_DIR}')
except ImportError:
    print('Not running in Colab — checkpoints saved locally to', CHECKPOINT_DIR)

Mounted at /content/drive
Drive mounted. Checkpoints will be saved to: /content/drive/MyDrive/esm2_checkpoints_690U


---
## Section 3: Load DGEB Datasets

DGEB datasets are loaded directly via the `dgeb` package's task classes (`ArchRetrieval`, `EukRetrieval`), which internally reference the correct HuggingFace dataset IDs with pinned revision hashes:

| Task | HF dataset (sequences) | HF dataset (qrels) |
|---|---|---|
| **Arch Retrieval** | `tattabio/arch_retrieval` | `tattabio/arch_retrieval_qrels` |
| **Euk Retrieval** | `tattabio/euk_retrieval` | `tattabio/euk_retrieval_qrels` |

**Column names:** `Entry` (UniProt ID), `Sequence` (amino acid string)  
**Splits:** `train` = bacterial corpus · `test` = archaeal/eukaryotic queries  
**Relevance:** `fuzz_ratio >= 90` → annotation similarity ≥ 0.9 → `y = 1`

In [6]:
# ---------------------------------------------------------------------------
# Load DGEB datasets using the dgeb package's own task definitions.
# This is the most reliable approach — it uses the exact dataset IDs and
# pinned revision hashes from the DGEB source code, avoiding any naming
# guesswork.
#
# Dataset structure (verified from dgeb/tasks/retrieval_tasks.py):
#   datasets[0]  ->  tattabio/arch_retrieval  (or euk_retrieval)
#                    split='train' = bacterial corpus  (columns: Entry, Sequence)
#                    split='test'  = archaeal/euk queries (columns: Entry, Sequence)
#   datasets[1]  ->  tattabio/arch_retrieval_qrels  (or euk_retrieval_qrels)
#                    columns: query_id, corpus_id, fuzz_ratio (int 0–100)
#                    pairs with fuzz_ratio >= 90 are relevant (annotation sim >= 0.9)
# ---------------------------------------------------------------------------
from dgeb.tasks import ArchRetrieval, EukRetrieval
from collections import defaultdict


def load_dgeb_task(task_class, task_name):
    """Load queries, corpus and qrels for a DGEB retrieval task.

    Uses the dgeb package's own metadata (pinned HF revision hashes) so the
    correct dataset version is always fetched.

    Returns
    -------
    queries : dict[str, str]  — {UniProt Entry: amino_acid_sequence}
    corpus  : dict[str, str]  — {UniProt Entry: amino_acid_sequence}
    qrels   : dict[str, set]  — {query_entry: set of relevant corpus_entries}
    """
    print(f'Loading {task_name} ...')
    task = task_class()
    meta = task.metadata

    # datasets[0] has corpus (train) and queries (test)
    data_ds  = meta.datasets[0].load()   # DatasetDict
    qrels_ds = meta.datasets[1].load()   # DatasetDict (typically single 'train' split)

    corpus  = {row['Entry']: row['Sequence'] for row in data_ds['train']}
    queries = {row['Entry']: row['Sequence'] for row in data_ds['test']}

    # Build binary qrels: threshold fuzz_ratio >= 90 ↔ annotation similarity >= 0.9
    qrels = defaultdict(set)
    # qrels_ds may be a DatasetDict (iterate all splits) or a Dataset
    splits = list(qrels_ds.values()) if hasattr(qrels_ds, 'values') else [qrels_ds]
    for split in splits:
        for row in split:
            if int(row['fuzz_ratio']) >= 90:
                qrels[str(row['query_id'])].add(str(row['corpus_id']))

    print(f'  Corpus  : {len(corpus):,}')
    print(f'  Queries : {len(queries):,}')
    print(f'  Queries with ≥1 relevant doc: {len(qrels):,}')
    return queries, corpus, dict(qrels)


arch_queries, arch_corpus, arch_qrels = load_dgeb_task(ArchRetrieval, 'Arch Retrieval')
euk_queries,  euk_corpus,  euk_qrels  = load_dgeb_task(EukRetrieval,  'Euk Retrieval')

Loading Arch Retrieval ...


README.md:   0%|          | 0.00/452 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/832k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9229 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/311 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/352 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29377 [00:00<?, ? examples/s]

  Corpus  : 3,202
  Queries : 311
  Queries with ≥1 relevant doc: 0


In [7]:
# Quick sanity check on sequence lengths
def seq_length_stats(seq_dict, name):
    lengths = [len(v) for v in seq_dict.values()]
    print(f'{name}: n={len(lengths):,}  '
          f'min={min(lengths)}  median={int(np.median(lengths))}  '
          f'mean={int(np.mean(lengths))}  max={max(lengths)}')

seq_length_stats(arch_queries, 'Arch queries')
seq_length_stats(arch_corpus,  'Arch corpus ')
seq_length_stats(euk_queries,  'Euk  queries')
seq_length_stats(euk_corpus,   'Euk  corpus ')

Arch queries: n=2,343  min=53  median=280  mean=331  max=1370
Arch corpus : n=9,229  min=54  median=289  mean=344  max=1308
Euk  queries: n=311  min=56  median=310  mean=366  max=1264
Euk  corpus : n=3,202  min=54  median=301  mean=353  max=1290


---
## Section 4: ESM2 Embedding Extraction

For each model scale we extract **both middle and last encoder layer** embeddings via mean-pooling, following DGEB methodology.

| Model HuggingFace ID | Params | Layers | Middle layer | Emb dim |
|---|---|---|---|---|
| `facebook/esm2_t12_35M_UR50D` | 35M | 12 | 6 | 480 |
| `facebook/esm2_t30_150M_UR50D` | 150M | 30 | 15 | 640 |
| `facebook/esm2_t33_650M_UR50D` | 650M | 33 | 17 | 1280 |

Embeddings are saved as `.npy` files keyed by `{model_size}_{layer}_{task}`. Load from checkpoint if already computed.

In [8]:
# Model registry — ordered smallest to largest
ESM2_MODELS = {
    '35M':  {
        'hf_id':        'facebook/esm2_t12_35M_UR50D',
        'num_layers':   12,
        'middle_layer': 6,
        'emb_dim':      480,
        'batch_size':   32,    # safe on T4/A100
        'max_length':   1024,
    },
    '150M': {
        'hf_id':        'facebook/esm2_t30_150M_UR50D',
        'num_layers':   30,
        'middle_layer': 15,
        'emb_dim':      640,
        'batch_size':   16,
        'max_length':   1024,
    },
    '650M': {
        'hf_id':        'facebook/esm2_t33_650M_UR50D',
        'num_layers':   33,
        'middle_layer': 17,
        'emb_dim':      1280,
        'batch_size':   4,     # conservative for A100 with long seqs
        'max_length':   1024,
    },
}

# ESM2 tokenizer uses <cls> and <eos> sentinel tokens — we mask them during mean-pooling
SENTINEL_TOKEN_IDS = None   # determined per-model after tokenizer load

In [11]:
def mean_pool(hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Mean-pool token hidden states, excluding padding tokens.

    Parameters
    ----------
    hidden_state    : (batch, seq_len, hidden_dim)
    attention_mask  : (batch, seq_len)  — 1 for real tokens, 0 for padding

    Returns
    -------
    pooled : (batch, hidden_dim)
    """
    mask = attention_mask.unsqueeze(-1).float()          # (B, L, 1)
    summed = (hidden_state * mask).sum(dim=1)            # (B, H)
    counts = mask.sum(dim=1).clamp(min=1e-9)             # (B, 1)
    return summed / counts


@torch.inference_mode()
def extract_embeddings(
    sequences: dict,
    model_cfg: dict,
    task_label: str,
    split_label: str,
    size_label: str,
    force_recompute: bool = False,
) -> dict:
    """Extract ESM2 embeddings (middle + last layer) for a dict of sequences.

    Saves/loads from CHECKPOINT_DIR to survive Colab disconnects.

    Parameters
    ----------
    sequences    : {id: aa_sequence}
    model_cfg    : entry from ESM2_MODELS
    task_label   : 'arch' or 'euk'
    split_label  : 'queries' or 'corpus'
    size_label   : '35M', '150M', '650M'
    force_recompute : skip cache and re-run

    Returns
    -------
    {
      'ids'    : list[str],
      'middle' : np.ndarray (N, emb_dim),   # middle encoder layer
      'last'   : np.ndarray (N, emb_dim),   # last  encoder layer
    }
    """
    ckpt_path = os.path.join(CHECKPOINT_DIR,
                             f'emb_{size_label}_{task_label}_{split_label}.pkl')

    if not force_recompute and os.path.exists(ckpt_path):
        print(f'  [CACHE HIT] Loading from {ckpt_path}')
        with open(ckpt_path, 'rb') as f:
            return pickle.load(f)

    hf_id        = model_cfg['hf_id']
    middle_layer = model_cfg['middle_layer']
    last_layer   = model_cfg['num_layers']     # transformers uses 1-indexed hidden states
    batch_size   = model_cfg['batch_size']
    max_length   = model_cfg['max_length']

    print(f'  Loading tokenizer & model: {hf_id}')
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModel.from_pretrained(hf_id, output_hidden_states=True)
    model = model.to(DEVICE).eval()

    ids = list(sequences.keys())
    seqs = [sequences[i] for i in ids]

    all_middle, all_last = [], []

    for batch_start in tqdm(range(0, len(seqs), batch_size),
                            desc=f'ESM2-{size_label} {task_label}/{split_label}'):
        batch_seqs = seqs[batch_start: batch_start + batch_size]
        enc = tokenizer(
            batch_seqs,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(DEVICE)

        outputs = model(**enc)
        # outputs.hidden_states: tuple of (num_layers+1) tensors of shape (B, L, H)
        # index 0 = embedding layer; index k = output of encoder layer k
        hidden_states = outputs.hidden_states

        mid_emb  = mean_pool(hidden_states[middle_layer], enc['attention_mask'])
        last_emb = mean_pool(hidden_states[last_layer],   enc['attention_mask'])

        all_middle.append(mid_emb.cpu().float().numpy())
        all_last.append(last_emb.cpu().float().numpy())

    # Cleanup GPU memory before next model
    del model
    torch.cuda.empty_cache()

    result = {
        'ids':    ids,
        'middle': np.vstack(all_middle),
        'last':   np.vstack(all_last),
    }

    with open(ckpt_path, 'wb') as f:
        pickle.dump(result, f, protocol=4)
    print(f'  Saved checkpoint: {ckpt_path}')
    return result

In [12]:
# ============================================================
# ESM2-35M   (fastest — run first to verify pipeline end-to-end)
# ============================================================
print('=== ESM2-35M ===')
cfg_35m = ESM2_MODELS['35M']

arch_query_emb_35m  = extract_embeddings(arch_queries, cfg_35m, 'arch', 'queries', '35M')
arch_corpus_emb_35m = extract_embeddings(arch_corpus,  cfg_35m, 'arch', 'corpus',  '35M')
euk_query_emb_35m   = extract_embeddings(euk_queries,  cfg_35m, 'euk',  'queries', '35M')
euk_corpus_emb_35m  = extract_embeddings(euk_corpus,   cfg_35m, 'euk',  'corpus',  '35M')

print('Middle layer shapes — arch queries:', arch_query_emb_35m['middle'].shape)
print('Last   layer shapes — arch corpus: ', arch_corpus_emb_35m['last'].shape)

=== ESM2-35M ===
  Loading tokenizer & model: facebook/esm2_t12_35M_UR50D


config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

  Saved checkpoint: /content/drive/MyDrive/esm2_checkpoints_690U/emb_35M_euk_queries.pkl
  Loading tokenizer & model: facebook/esm2_t12_35M_UR50D


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM2-35M euk/corpus:   0%|          | 0/101 [00:00<?, ?it/s]

  Saved checkpoint: /content/drive/MyDrive/esm2_checkpoints_690U/emb_35M_euk_corpus.pkl
Middle layer shapes — arch queries: (2343, 480)
Last   layer shapes — arch corpus:  (9229, 480)


In [13]:
# ============================================================
# ESM2-150M
# ============================================================
print('=== ESM2-150M ===')
cfg_150m = ESM2_MODELS['150M']

arch_query_emb_150m  = extract_embeddings(arch_queries, cfg_150m, 'arch', 'queries', '150M')
arch_corpus_emb_150m = extract_embeddings(arch_corpus,  cfg_150m, 'arch', 'corpus',  '150M')
euk_query_emb_150m   = extract_embeddings(euk_queries,  cfg_150m, 'euk',  'queries', '150M')
euk_corpus_emb_150m  = extract_embeddings(euk_corpus,   cfg_150m, 'euk',  'corpus',  '150M')

=== ESM2-150M ===
  Loading tokenizer & model: facebook/esm2_t30_150M_UR50D


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

  Saved checkpoint: /content/drive/MyDrive/esm2_checkpoints_690U/emb_150M_euk_queries.pkl
  Loading tokenizer & model: facebook/esm2_t30_150M_UR50D


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM2-150M euk/corpus:   0%|          | 0/201 [00:00<?, ?it/s]

  Saved checkpoint: /content/drive/MyDrive/esm2_checkpoints_690U/emb_150M_euk_corpus.pkl


In [14]:
# ============================================================
# ESM2-650M  (requires A100 on Colab Pro for stability)
# Sequences are truncated to 1024 tokens; mixed-precision saves ~40% VRAM
# ============================================================
print('=== ESM2-650M ===')

# Enable automatic mixed precision (bfloat16) to reduce VRAM for 650M
original_dtype = torch.get_default_dtype()

cfg_650m = ESM2_MODELS['650M']

arch_query_emb_650m  = extract_embeddings(arch_queries, cfg_650m, 'arch', 'queries', '650M')
arch_corpus_emb_650m = extract_embeddings(arch_corpus,  cfg_650m, 'arch', 'corpus',  '650M')
euk_query_emb_650m   = extract_embeddings(euk_queries,  cfg_650m, 'euk',  'queries', '650M')
euk_corpus_emb_650m  = extract_embeddings(euk_corpus,   cfg_650m, 'euk',  'corpus',  '650M')

=== ESM2-650M ===
  Loading tokenizer & model: facebook/esm2_t33_650M_UR50D


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

  Saved checkpoint: /content/drive/MyDrive/esm2_checkpoints_690U/emb_650M_euk_queries.pkl
  Loading tokenizer & model: facebook/esm2_t33_650M_UR50D


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM2-650M euk/corpus:   0%|          | 0/801 [00:00<?, ?it/s]

  Saved checkpoint: /content/drive/MyDrive/esm2_checkpoints_690U/emb_650M_euk_corpus.pkl


---
## Section 5: Cosine Similarity Retrieval (Top-5)

For each query we rank all corpus proteins by cosine similarity and return the top 5. This is identical to the retrieval step in Method 2 (k-mer), enabling direct comparison.

In [ ]:
def cosine_retrieve(
    query_embs: np.ndarray,
    corpus_embs: np.ndarray,
    corpus_ids: list,
    top_k: int = 5,
    chunk_size: int = 256,
) -> list:
    """Retrieve top-k corpus proteins for each query by cosine similarity.

    Uses chunked matrix multiplication to avoid OOM on large corpora.

    Parameters
    ----------
    query_embs  : (Q, D) — L2-normalised query embeddings
    corpus_embs : (C, D) — L2-normalised corpus embeddings
    corpus_ids  : list of length C — corpus protein IDs
    top_k       : number of results to return per query
    chunk_size  : number of queries to score in one batch

    Returns
    -------
    results : list of lists  [[corpus_id, ...], ...]  length Q
    """
    # L2-normalise embeddings for dot-product ≡ cosine similarity
    q_norm = query_embs  / (np.linalg.norm(query_embs,  axis=1, keepdims=True) + 1e-10)
    c_norm = corpus_embs / (np.linalg.norm(corpus_embs, axis=1, keepdims=True) + 1e-10)
    corpus_ids_arr = np.array(corpus_ids)

    all_results = []
    for start in tqdm(range(0, len(q_norm), chunk_size), desc='Retrieval'):
        q_chunk = q_norm[start: start + chunk_size]    # (chunk, D)
        sims    = q_chunk @ c_norm.T                   # (chunk, C)
        top_idx = np.argpartition(sims, -top_k, axis=1)[:, -top_k:]
        # Sort within top-k by descending similarity
        for i, row_idx in enumerate(top_idx):
            row_sims = sims[i, row_idx]
            sorted_local = row_idx[np.argsort(-row_sims)]
            all_results.append(corpus_ids_arr[sorted_local].tolist())

    return all_results

In [ ]:
# Build retrieval results for all (model × layer × task) combinations
# Each key: (model_size, layer, task)  ->  list of top-5 corpus_id lists
RETRIEVAL_RESULTS = {}

TASK_CONFIG = {
    'arch': {
        'query_ids':  arch_query_emb_35m['ids'],   # same for all models
        'corpus_ids': arch_corpus_emb_35m['ids'],
        'qrels':      arch_qrels,
        'embs': {
            '35M':  (arch_query_emb_35m,  arch_corpus_emb_35m),
            '150M': (arch_query_emb_150m, arch_corpus_emb_150m),
            '650M': (arch_query_emb_650m, arch_corpus_emb_650m),
        },
    },
    'euk': {
        'query_ids':  euk_query_emb_35m['ids'],
        'corpus_ids': euk_corpus_emb_35m['ids'],
        'qrels':      euk_qrels,
        'embs': {
            '35M':  (euk_query_emb_35m,  euk_corpus_emb_35m),
            '150M': (euk_query_emb_150m, euk_corpus_emb_150m),
            '650M': (euk_query_emb_650m, euk_corpus_emb_650m),
        },
    },
}

for task, tcfg in TASK_CONFIG.items():
    corpus_ids = tcfg['corpus_ids']
    for size, (qemb_dict, cemb_dict) in tcfg['embs'].items():
        for layer in ('middle', 'last'):
            key = (size, layer, task)
            print(f'Retrieving: model={size}  layer={layer}  task={task}')
            RETRIEVAL_RESULTS[key] = cosine_retrieve(
                qemb_dict[layer], cemb_dict[layer], corpus_ids, top_k=5
            )

print('Retrieval complete for', len(RETRIEVAL_RESULTS), 'configurations.')

---
## Section 6: Evaluation — MAP@5, nDCG@5, Recall@5

All metrics are computed per-query then macro-averaged, matching DGEB's evaluation protocol.

In [ ]:
def average_precision_at_k(retrieved: list, relevant: set, k: int = 5) -> float:
    """Compute Average Precision@k for a single query."""
    if not relevant:
        return 0.0
    hits, score = 0, 0.0
    for rank, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant:
            hits += 1
            score += hits / rank
    return score / min(len(relevant), k)


def ndcg_at_k(retrieved: list, relevant: set, k: int = 5) -> float:
    """Compute nDCG@k for a single query (binary relevance)."""
    if not relevant:
        return 0.0
    dcg = sum(
        (1.0 if doc_id in relevant else 0.0) / np.log2(rank + 1)
        for rank, doc_id in enumerate(retrieved[:k], start=1)
    )
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def recall_at_k(retrieved: list, relevant: set, k: int = 5) -> float:
    """Compute Recall@k for a single query."""
    if not relevant:
        return 0.0
    hits = sum(1 for doc_id in retrieved[:k] if doc_id in relevant)
    return hits / len(relevant)


def evaluate_retrieval(
    query_ids: list,
    all_retrieved: list,
    qrels: dict,
    k: int = 5,
) -> dict:
    """Compute MAP@k, nDCG@k, Recall@k across all queries.

    Queries without any relevant document are excluded from the average
    (consistent with DGEB evaluation).
    """
    maps, ndcgs, recalls = [], [], []
    for qid, retrieved in zip(query_ids, all_retrieved):
        relevant = qrels.get(qid, set())
        if not relevant:
            continue
        maps.append(average_precision_at_k(retrieved, relevant, k))
        ndcgs.append(ndcg_at_k(retrieved, relevant, k))
        recalls.append(recall_at_k(retrieved, relevant, k))

    return {
        f'MAP@{k}':    float(np.mean(maps))    if maps else 0.0,
        f'nDCG@{k}':   float(np.mean(ndcgs))   if ndcgs else 0.0,
        f'Recall@{k}': float(np.mean(recalls)) if recalls else 0.0,
        'n_queries':   len(maps),
    }

In [ ]:
# Evaluate all configurations and collect results
EVAL_RESULTS = []

for (size, layer, task), retrieved_list in RETRIEVAL_RESULTS.items():
    tcfg     = TASK_CONFIG[task]
    query_ids = tcfg['query_ids']
    qrels     = tcfg['qrels']
    metrics   = evaluate_retrieval(query_ids, retrieved_list, qrels, k=5)
    EVAL_RESULTS.append({
        'Method':   f'ESM2-{size}',
        'Layer':    layer,
        'Task':     task.capitalize() + ' Retrieval',
        **metrics,
    })

df_results = pd.DataFrame(EVAL_RESULTS)
df_results = df_results.sort_values(['Task', 'Method', 'Layer']).reset_index(drop=True)

# Format for display
display_cols = ['Method', 'Layer', 'Task', 'MAP@5', 'nDCG@5', 'Recall@5', 'n_queries']
print(df_results[display_cols].to_string(index=False, float_format='%.4f'))

In [ ]:
# For each (model, task) pick the best-of-{middle, last} by MAP@5, per DGEB methodology
df_best = (
    df_results
    .sort_values('MAP@5', ascending=False)
    .groupby(['Method', 'Task'], as_index=False)
    .first()
)
df_best = df_best.sort_values(['Task', 'Method']).reset_index(drop=True)
print('\n=== Best layer per model (DGEB protocol: report best of middle vs last) ===')
print(df_best[display_cols].to_string(index=False, float_format='%.4f'))

# Save results to CSV
df_results.to_csv(os.path.join(CHECKPOINT_DIR, 'esm2_all_results.csv'), index=False)
df_best.to_csv(os.path.join(CHECKPOINT_DIR, 'esm2_best_results.csv'), index=False)
print('\nResults saved to CSV.')

---
## Section 7: Twilight Zone Analysis

We bin query–corpus pairs by BLAST percent identity (`%ID`) and compute MAP@5 separately for:
- **High similarity**: %ID > 40%  
- **Medium similarity**: 20% ≤ %ID ≤ 40%  
- **Twilight zone**: %ID < 20%  

This analysis tests whether ESM2 provides gains where BLASTp is known to struggle.  

> **Prerequisite**: BLAST percent identity values must come from **Method 1** (BLASTp notebook).  
> Upload the output file `blast_pident.csv` from Method 1, with columns `[query_id, corpus_id, pident]`.

In [ ]:
# Load BLAST percent identity data produced by Method 1
# Expected CSV: query_id, corpus_id, pident  (one row per top BLAST hit)
BLAST_PIDENT_PATH = os.path.join(CHECKPOINT_DIR, 'blast_pident.csv')

if os.path.exists(BLAST_PIDENT_PATH):
    df_blast = pd.read_csv(BLAST_PIDENT_PATH)
    print(f'Loaded BLAST pident data: {len(df_blast):,} rows')
    print(df_blast.head())
else:
    print('blast_pident.csv not found.')
    print('Please upload the output from Method 1 to:', BLAST_PIDENT_PATH)
    print('Twilight zone analysis will be skipped.')
    df_blast = None

In [ ]:
def bin_by_pident(pident: float) -> str:
    if pident >= 40:  return '>40% (High)'
    if pident >= 20:  return '20-40% (Medium)'
    return '<20% (Twilight)'


def twilight_zone_map(
    query_ids: list,
    retrieved_list: list,
    qrels: dict,
    df_blast_pident: pd.DataFrame,
    k: int = 5,
) -> dict:
    """Compute MAP@k per BLAST %ID bin.

    For each query we find the maximum %ID across its corpus hits (best BLAST alignment).
    Queries with no BLAST hit are assigned pident=0 (twilight zone).
    """
    # Build lookup: query_id -> max pident across all corpus hits
    max_pident = (
        df_blast_pident
        .groupby('query_id')['pident']
        .max()
        .to_dict()
    )

    bin_aps = {'>40% (High)': [], '20-40% (Medium)': [], '<20% (Twilight)': []}

    for qid, retrieved in zip(query_ids, retrieved_list):
        relevant = qrels.get(qid, set())
        if not relevant:
            continue
        pident = max_pident.get(qid, 0.0)
        bin_label = bin_by_pident(pident)
        ap = average_precision_at_k(retrieved, relevant, k)
        bin_aps[bin_label].append(ap)

    return {
        bin_label: (float(np.mean(aps)) if aps else np.nan, len(aps))
        for bin_label, aps in bin_aps.items()
    }


if df_blast is not None:
    TZ_RESULTS = {}
    # Evaluate best ESM2 config per model on Arch retrieval
    for size in ('35M', '150M', '650M'):
        best_layer_row = df_best[
            (df_best['Method'] == f'ESM2-{size}') &
            (df_best['Task'] == 'Arch Retrieval')
        ]
        if best_layer_row.empty:
            continue
        best_layer = best_layer_row.iloc[0]['Layer']
        retrieved = RETRIEVAL_RESULTS[(size, best_layer, 'arch')]
        tz = twilight_zone_map(
            TASK_CONFIG['arch']['query_ids'],
            retrieved,
            arch_qrels,
            df_blast[df_blast['task'] == 'arch'] if 'task' in df_blast.columns else df_blast,
        )
        TZ_RESULTS[f'ESM2-{size}'] = tz
        print(f'ESM2-{size} (Arch):')
        for bin_lbl, (score, n) in tz.items():
            print(f'  {bin_lbl:22s}: MAP@5={score:.4f}  n={n}')
else:
    print('Skipping twilight zone analysis (no BLAST data).')

---
## Section 8: Visualization & Results Table

Three outputs:
1. **Table 1** — Main comparison table (all methods, both tasks)  
2. **Figure 1** — Scaling plot: MAP@5 vs. parameter count  
3. **Figure 2** — Twilight zone grouped bar chart  

In [ ]:
plt.rcParams.update({
    'figure.dpi':      150,
    'font.family':     'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid':       True,
    'grid.alpha':      0.3,
})
COLORS = sns.color_palette('tab10')

In [ ]:
# ----------------------------------------------------------------
# Table 1 — Main comparison (Method 3 rows only; Methods 1 & 2
# will be filled in from the other notebooks when consolidating)
# ----------------------------------------------------------------
DGEB_3B_REF = [
    {'Method': 'ESM2-3B', 'Layer': 'best (DGEB reported)', 'Task': 'Arch Retrieval',
     'MAP@5': 0.313, 'nDCG@5': float('nan'), 'Recall@5': float('nan'), 'n_queries': 'N/A'},
    {'Method': 'ESM2-3B', 'Layer': 'best (DGEB reported)', 'Task': 'Euk Retrieval',
     'MAP@5': 0.357, 'nDCG@5': float('nan'), 'Recall@5': float('nan'), 'n_queries': 'N/A'},
]

df_table1 = pd.concat([
    df_best[display_cols],
    pd.DataFrame(DGEB_3B_REF)[display_cols]
], ignore_index=True)

print('\n=== TABLE 1 — ESM2 Results Summary (Method 3) ===')
print(df_table1.to_string(index=False, float_format='%.4f'))

In [ ]:
# ----------------------------------------------------------------
# Figure 1 — Scaling plot: MAP@5 vs. parameter count
# ----------------------------------------------------------------
PARAM_COUNTS = {'35M': 35e6, '150M': 150e6, '650M': 650e6, '3B': 3e9}

# Placeholder values for Methods 1 & 2 — replace with actual results
BLASTP_MAP   = {'arch': None, 'euk': None}   # fill from Method 1 notebook
KMER_MAP     = {'arch': None, 'euk': None}   # fill from Method 2 notebook

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for ax, task in zip(axes, ('arch', 'euk')):
    task_label = task.capitalize() + ' Retrieval'

    # ESM2 scaling line
    esm2_rows = df_best[df_best['Task'] == task_label].copy()
    esm2_rows['params'] = esm2_rows['Method'].str.extract(r'(\d+M|\d+B)')[0].map(PARAM_COUNTS)
    esm2_rows = esm2_rows.dropna(subset=['params']).sort_values('params')

    # Add 3B reference point
    ref_map = 0.313 if task == 'arch' else 0.357
    esm2_params = list(esm2_rows['params']) + [3e9]
    esm2_maps   = list(esm2_rows['MAP@5']) + [ref_map]

    ax.plot(esm2_params, esm2_maps, 'o-', color=COLORS[0], label='ESM2 (ours)', linewidth=2)
    ax.scatter([3e9], [ref_map], marker='*', s=200, color=COLORS[0],
               label='ESM2-3B (DGEB paper)', zorder=5)

    # Baseline dashed lines (fill in once Methods 1 & 2 are run)
    if BLASTP_MAP[task] is not None:
        ax.axhline(BLASTP_MAP[task], linestyle='--', color=COLORS[1], label='BLASTp')
    if KMER_MAP[task] is not None:
        ax.axhline(KMER_MAP[task], linestyle=':', color=COLORS[2], label='Best k-mer')

    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{int(x/1e6)}M' if x < 1e9 else f'{x/1e9:.0f}B'))
    ax.set_xlabel('ESM2 Parameters', fontsize=11)
    ax.set_ylabel('MAP@5', fontsize=11)
    ax.set_title(task_label, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Figure 1 — ESM2 Scaling: MAP@5 vs. Model Size', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(CHECKPOINT_DIR, 'fig1_scaling_plot.png'), bbox_inches='tight')
plt.show()
print('Saved: fig1_scaling_plot.png')

In [ ]:
# ----------------------------------------------------------------
# Figure 2 — Twilight zone grouped bar chart (Arch Retrieval)
# ----------------------------------------------------------------
if df_blast is not None and TZ_RESULTS:
    BIN_ORDER = ['<20% (Twilight)', '20-40% (Medium)', '>40% (High)']
    method_names = list(TZ_RESULTS.keys())
    x = np.arange(len(BIN_ORDER))
    width = 0.22

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (method, tz_data) in enumerate(TZ_RESULTS.items()):
        scores = [tz_data[b][0] for b in BIN_ORDER]
        ns     = [tz_data[b][1] for b in BIN_ORDER]
        offset = (i - len(method_names) / 2 + 0.5) * width
        bars = ax.bar(x + offset, scores, width, label=method, color=COLORS[i], alpha=0.85)
        for bar, n in zip(bars, ns):
            if bar.get_height() > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                        f'n={n}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(BIN_ORDER, fontsize=10)
    ax.set_ylabel('MAP@5', fontsize=11)
    ax.set_title('Figure 2 — Twilight Zone Analysis (Arch Retrieval)\nMAP@5 by BLAST % Identity Bin',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    fig.savefig(os.path.join(CHECKPOINT_DIR, 'fig2_twilight_zone.png'), bbox_inches='tight')
    plt.show()
    print('Saved: fig2_twilight_zone.png')
else:
    print('Twilight zone figure skipped (no BLAST pident data).')
    print('Re-run Section 7 after uploading blast_pident.csv from Method 1.')

In [ ]:
# ----------------------------------------------------------------
# Supplementary — Middle vs Last Layer comparison
# ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, task_label in zip(axes, ('Arch Retrieval', 'Euk Retrieval')):
    task_df = df_results[df_results['Task'] == task_label].copy()
    sizes = ['35M', '150M', '650M']
    x = np.arange(len(sizes))
    width = 0.35

    for j, layer in enumerate(('middle', 'last')):
        layer_df = task_df[task_df['Layer'] == layer].set_index('Method')
        scores = [layer_df.loc[f'ESM2-{s}', 'MAP@5'] if f'ESM2-{s}' in layer_df.index
                  else 0.0 for s in sizes]
        offset = (j - 0.5) * width
        ax.bar(x + offset, scores, width, label=f'{layer.capitalize()} layer',
               color=COLORS[j + 3], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([f'ESM2-{s}' for s in sizes])
    ax.set_ylabel('MAP@5')
    ax.set_title(f'{task_label}\nMiddle vs. Last Encoder Layer', fontweight='bold')
    ax.legend()

fig.suptitle('Supplementary — Layer Selection Effect on MAP@5', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(CHECKPOINT_DIR, 'supp_layer_comparison.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ----------------------------------------------------------------
# Final summary printout for paper Table 1
# ----------------------------------------------------------------
print('=' * 70)
print('FINAL RESULTS — ESM2 Embeddings (Method 3)')
print('=' * 70)
for task_label in ('Arch Retrieval', 'Euk Retrieval'):
    print(f'\n{task_label}:')
    t = df_best[df_best['Task'] == task_label][['Method', 'Layer', 'MAP@5', 'nDCG@5', 'Recall@5']]
    print(t.to_string(index=False, float_format='%.4f'))
    print(f'  ESM2-3B (DGEB paper ref): MAP@5 = {0.313 if "Arch" in task_label else 0.357}')

print('\nAll outputs saved to:', CHECKPOINT_DIR)